In [1]:
#import
import pandas as pd
from sklearn.model_selection import train_test_split  #データの分割

from sklearn.preprocessing import StandardScaler #標準化
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰
from sklearn.linear_model import Lasso  #ラッソ回帰

In [2]:
df = pd.read_csv('datafiles/train.csv')

In [3]:
#欠損値の確認
for c in df.columns:
    null_counts = df[c].isnull().sum()
    if null_counts != 0:
        print(f'{null_counts}  列＝{c}')

259  列＝LotFrontage
1369  列＝Alley
872  列＝MasVnrType
8  列＝MasVnrArea
37  列＝BsmtQual
37  列＝BsmtCond
38  列＝BsmtExposure
37  列＝BsmtFinType1
38  列＝BsmtFinType2
1  列＝Electrical
690  列＝FireplaceQu
81  列＝GarageType
81  列＝GarageYrBlt
81  列＝GarageFinish
81  列＝GarageQual
81  列＝GarageCond
1453  列＝PoolQC
1179  列＝Fence
1406  列＝MiscFeature


In [4]:
#明らかに不要な'id'を除く
df = df.drop(['Id'], axis = 1)

In [5]:
#ダミー変数化する行の抜き出し
to_dummy_cols = []
for c in df.columns:
    if type(c) == str:
        to_dummy_cols.append(c)
print(to_dummy_cols)

['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'PoolQC', 'Fen

In [6]:
'''
#strの特徴量の中にNAが混ざっている列
Alley
MasVnrType
BsmtQual
BsmtCond
BsmtExposure
BsmtFinType1
BsmtFinType2
Electrical
FireplaceQu
GarageType
GarageFinish
GarageQual
GarageCond
PoolQC
Fence
MiscFeature

#intの特徴量の中にNAが混ざっている列
LotFrontage  NAを0に変更
MasVnrArea   NAを0に変更
GarageYrBlt  NAを0に変更
'''
#data_description.txt を確認すると、すべての特徴量で'NA'に意味があるようだったので補完
to_NA_cols = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'Electrical', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 
    'GarageCond', 'Fence', 'MiscFeature'
]
df[to_NA_cols] = df[to_NA_cols].fillna('NA')
df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']] = df[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']].fillna(0.0)


In [7]:
print(df.columns)

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'Wo

In [8]:
#float型に変更
not_to_dummy = ['MSSubClass', 'LotFrontage', 
    'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 
    'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 
    'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 
    'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 
    'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
    'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice'
]

In [9]:
df[not_to_dummy] = df[not_to_dummy].astype('float64')

In [10]:
#ダミー変数化
to_dummy = set(df.columns) - set(not_to_dummy)
to_dummy = list(to_dummy)
for c in to_dummy:
    dummy = pd.get_dummies(df[c], drop_first = True, dtype = int)
    df = pd.concat([df, dummy], axis = 1)
    df = df.drop([c], axis = 1)

In [11]:
#float型に変更
df = df.astype('float64')

In [12]:
#説明変数と目的変数の指定
x_cols=[c for c in df.columns if c != 'SalePrice']
y_cols=['SalePrice']

#標準化
sc_model=StandardScaler()
sc_model.fit(df[x_cols])

,copy,True
,with_mean,True
,with_std,True


In [13]:
#重回帰、リッジ回帰、ラッソ回帰、回帰木を実践し、結果を比較する。
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [14]:
#重回帰
model1 = LinearRegression()
result = cross_validate(model1,df[x_cols], df[y_cols], cv = kf, scoring = 'r2', return_train_score = True)
print(sum(result['test_score'])/len(result['test_score']))

0.5360101293105316


In [ ]:
#リッジ回帰
#正則化項の定数を0.01~20まで検証
best_ridgescore = 0
best_alpha = 0

#alpha（Fの係数）を1~100まで変化させて実験
for i in range(1,101):
    num = i
    ridgeModel = Ridge(random_state = 0,alpha = num)
    all_result = cross_validate(ridgeModel,df[x_cols], df[y_cols], cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_ridgescore:
        best_ridgescore = result
        best_alpha = num
print(f'正則化項＝{best_alpha}　リッジ回帰のスコア＝{best_ridgescore}')

#完成したリッジ回帰モデルで学習
model2 = Ridge(alpha = (best_alpha/100))
result = cross_validate(model2,df[x_cols], df[y_cols], cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したモデルのスコア＝{sum(result['test_score'])/len(result['test_score'])}')

正則化項＝56　リッジ回帰のスコア＝0.831605226452021
完成したモデルのスコア＝0.7213034077047203


In [23]:
#ラッソ回帰
#正則化項の定数を0.01~20まで検証
best_lassoscore = 0
best_alpha = 0
#alpha（Fの係数）を1~100まで変化させて実験
for i in range(1,101):
    num = i
    lassoModel = Lasso(random_state = 0, alpha = num)
    all_result = cross_validate(lassoModel,df[x_cols], df[y_cols], cv = kf, scoring = 'r2', return_train_score = True)
    result = sum(all_result['test_score'])/len(all_result['test_score'])
    if result > best_lassoscore:
        best_lassoscore = result
        best_alpha = num
print(f'正則化項＝{best_alpha}　ラッソ回帰のスコア＝{best_lassoscore}')

#完成したラッソ回帰モデルで学習
model3 = Lasso(alpha = (best_alpha/100))
result = cross_validate(model3,df[x_cols], df[y_cols], cv = kf, scoring = 'r2', return_train_score = True)
print(f'完成したモデルのスコア＝{sum(result['test_score'])/len(result['test_score'])}')


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.028e+11, tolerance: 7.191e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.490e+11, tolerance: 7.582e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.006e+11, toleranc

正則化項＝100　ラッソ回帰のスコア＝0.7817773750098567


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.028e+11, tolerance: 7.191e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.490e+11, tolerance: 7.582e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.006e+11, toleranc

完成したモデルのスコア＝0.5610263975714579


c:\Users\natsu\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.519e+11, tolerance: 7.732e+08
  model = cd_fast.enet_coordinate_descent(


In [ ]:
'''
単純なモデルでは、下記のような結果となった。

重回帰のスコア  0.5360101293105316
リッジ回帰のスコア  0.7213034077047203
ラッソ回帰のスコア  0.5610263975714579

以下では、特徴量を改善したモデルでも実験する。
'''